# Agent Failure Atlas — Main Analysis Notebook

This notebook performs the complete analysis for the Agent Failure Atlas (AFA) paper:
- Dataset loading and exploration
- Failure category distribution
- Cross-model comparison
- Statistical tests
- Early failure signal analysis
- All publication-ready figures

**Prerequisites:**
```bash
pip install -r experiments/requirements.txt
python dataset/generate_afad.py
```

In [ ]:
import sys
sys.path.insert(0, '..')  # Add project root

import json
import warnings
warnings.filterwarnings('ignore')
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 120})

print('Libraries loaded successfully.')

## 1. Load the AFAD Dataset

In [ ]:
def load_afad(filepath):
    records = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

# Load dataset (run dataset/generate_afad.py first if not present)
records = load_afad('../dataset/afad_v1.jsonl')
df = pd.DataFrame(records)

print(f'Loaded {len(records)} records')
print(f'Columns: {list(df.columns)}')
df.head(3)

## 2. Dataset Overview

In [ ]:
print('=== Dataset Overview ===')
print(f"Total records: {len(df)}")
print(f"\nBy Model:")
print(df['model'].value_counts().to_string())
print(f"\nBy Task Type:")
print(df['task_type'].value_counts().to_string())
print(f"\nBy Outcome:")
print(df['outcome'].value_counts().to_string())
print(f"\nOverall Failure Rate: {(df['outcome']=='failure').mean():.1%}")
print(f"Overall Recovery Rate: {df['recovered'].mean():.1%}")
print(f"Mean Severity: {df['severity_score'].mean():.2f}")

## 3. Failure Category Distribution

In [ ]:
CATEGORIES = ['PLAN', 'REAS', 'TOOL', 'MEM', 'EXEC', 'COOR', 'SAFE', 'ALIG']
CATEGORY_COLORS = {
    'PLAN': '#4C72B0', 'REAS': '#DD8452', 'TOOL': '#55A868',
    'MEM': '#C44E52', 'EXEC': '#8172B2', 'COOR': '#937860',
    'SAFE': '#DA8BC3', 'ALIG': '#8C8C8C',
}

cat_counts = df['failure_label'].value_counts()
cats_ordered = [c for c in CATEGORIES if c in cat_counts.index]
counts = [cat_counts[c] for c in cats_ordered]
colors = [CATEGORY_COLORS[c] for c in cats_ordered]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(cats_ordered, counts, color=colors, edgecolor='white', linewidth=0.5)
ax.bar_label(bars, padding=3)
ax.set_title('AFAD: Overall Failure Category Distribution', fontweight='bold', fontsize=14)
ax.set_xlabel('Failure Category')
ax.set_ylabel('Count')
ax.set_ylim(0, max(counts) * 1.15)

# Add category names as subtitle
cat_labels = {
    'PLAN': 'Planning', 'REAS': 'Reasoning', 'TOOL': 'Tool Use',
    'MEM': 'Memory', 'EXEC': 'Execution', 'COOR': 'Coordination',
    'SAFE': 'Safety', 'ALIG': 'Alignment'
}
ax.set_xticklabels([f"{c}\n({cat_labels[c]})" for c in cats_ordered])
plt.tight_layout()
plt.savefig('../analysis/results/figures/failure_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Cross-Model Comparison

In [ ]:
MODELS = ['GPT-OSS-20B', 'Qwen3-8B', 'Qwen3-30B', 'DeepSeek-R1-8B', 'Gemma3-12B', 'Llama-3.2']

model_stats = []
for model in MODELS:
    mdf = df[df['model'] == model]
    if len(mdf) == 0:
        continue
    failure_rate = (mdf['outcome'] == 'failure').mean()
    recovery_rate = mdf['recovered'].mean()
    mean_severity = mdf['severity_score'].mean()
    traj_lengths = mdf['trajectory'].apply(lambda t: len(t) if isinstance(t, list) else 0)
    failure_density = traj_lengths[mdf['outcome'] == 'failure'].mean()
    
    model_stats.append({
        'Model': model,
        'N': len(mdf),
        'Failure Rate': failure_rate,
        'Recovery Rate': recovery_rate,
        'Mean Severity': mean_severity,
        'Failure Density': failure_density,
    })

stats_df = pd.DataFrame(model_stats)
display_df = stats_df.copy()
display_df['Failure Rate'] = display_df['Failure Rate'].apply(lambda x: f'{x:.1%}')
display_df['Recovery Rate'] = display_df['Recovery Rate'].apply(lambda x: f'{x:.1%}')
display_df['Mean Severity'] = display_df['Mean Severity'].apply(lambda x: f'{x:.2f}')
display_df['Failure Density'] = display_df['Failure Density'].apply(lambda x: f'{x:.1f}')
print('Cross-Model Metrics Summary')
print(display_df.to_string(index=False))

In [ ]:
# Grouped bar chart
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

model_names_short = [m.replace('DeepSeek-R1-8B', 'DS-R1-8B') for m in stats_df['Model']]

for ax, col, color, title in [
    (axes[0], 'Failure Rate', '#C44E52', 'Failure Rate'),
    (axes[1], 'Recovery Rate', '#55A868', 'Recovery Rate'),
    (axes[2], 'Mean Severity', '#4C72B0', 'Mean Severity (1-5)'),
]:
    vals = stats_df[col].values
    bars = ax.barh(model_names_short, vals * (100 if col != 'Mean Severity' else 1),
                   color=color, edgecolor='white')
    ax.set_title(title, fontweight='bold')
    if col != 'Mean Severity':
        ax.set_xlabel('%')
        for i, v in enumerate(vals):
            ax.text(v*100 + 0.3, i, f'{v:.1%}', va='center', fontsize=10)
    else:
        ax.set_xlabel('Score')
        ax.set_xlim(0, 5)
        for i, v in enumerate(vals):
            ax.text(v + 0.05, i, f'{v:.2f}', va='center', fontsize=10)

plt.suptitle('AFA Benchmark — Cross-Model Failure Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../analysis/results/figures/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Failure Category Heatmap per Model

In [ ]:
import numpy as np

heatmap_data = []
model_rows = []
for model in MODELS:
    mdf = df[df['model'] == model]
    if len(mdf) == 0:
        continue
    row = [mdf[mdf['failure_label'] == cat].shape[0] / len(mdf) * 100 for cat in CATEGORIES]
    heatmap_data.append(row)
    model_rows.append(model)

hm_df = pd.DataFrame(heatmap_data, index=model_rows, columns=CATEGORIES)

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(
    hm_df, annot=True, fmt='.1f', cmap='YlOrRd',
    linewidths=0.5, ax=ax,
    cbar_kws={'label': '% of model records'}
)
ax.set_title('Failure Category Distribution per Model (%)', fontweight='bold', fontsize=14, pad=12)
ax.set_xlabel('Failure Category')
ax.set_ylabel('Model')
plt.tight_layout()
plt.savefig('../analysis/results/figures/category_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Severity Distribution

In [ ]:
severity_colors = {1: '#2ecc71', 2: '#f1c40f', 3: '#e67e22', 4: '#e74c3c', 5: '#8e44ad'}

fig, ax = plt.subplots(figsize=(12, 5))
bottoms = np.zeros(len(model_rows))

for sev in [1, 2, 3, 4, 5]:
    heights = []
    for model in model_rows:
        mdf = df[df['model'] == model]
        pct = (mdf['severity_score'] == sev).mean() * 100
        heights.append(pct)
    ax.bar(model_rows, heights, bottom=bottoms,
           color=severity_colors[sev], label=f'Severity {sev}',
           edgecolor='white', linewidth=0.5)
    bottoms += np.array(heights)

ax.set_title('Severity Score Distribution per Model', fontweight='bold', fontsize=14)
ax.set_ylabel('Percentage of Records')
ax.set_ylim(0, 110)
ax.legend(title='Severity', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig('../analysis/results/figures/severity_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Failure Subcategory Top-15

In [ ]:
top15 = df['failure_subcategory'].value_counts().head(15)
top15_labels = top15.index.tolist()
top15_colors = [CATEGORY_COLORS.get(l.split('-')[0], '#8C8C8C') for l in top15_labels]

fig, ax = plt.subplots(figsize=(9, 7))
bars = ax.barh(top15_labels[::-1], top15.values[::-1],
               color=top15_colors[::-1], edgecolor='white')
ax.bar_label(bars, padding=3)
ax.set_title('Top 15 Failure Subcategories', fontweight='bold', fontsize=14)
ax.set_xlabel('Count')
plt.tight_layout()
plt.savefig('../analysis/results/figures/subcategory_frequency.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Statistical Tests

In [ ]:
from scipy import stats

# Chi-square test: are failure rates different across models?
observed = []
for model in MODELS:
    mdf = df[df['model'] == model]
    if len(mdf) == 0:
        continue
    failures = (mdf['outcome'] == 'failure').sum()
    successes = len(mdf) - failures
    observed.append([failures, successes])

chi2, p, dof, expected = stats.chi2_contingency(observed)
print('Chi-Square Test: Failure Rates Across Models')
print(f'  Chi² = {chi2:.4f}, df = {dof}, p = {p:.6f}')
print(f'  Interpretation: {"Significant (p<0.05)" if p < 0.05 else "Not significant"}')

# Kruskal-Wallis test: severity distributions
severity_groups = [df[df['model'] == m]['severity_score'].dropna().tolist() for m in MODELS if m in df['model'].values]
H, p_kw = stats.kruskal(*[g for g in severity_groups if len(g) > 1])
print(f'\nKruskal-Wallis Test: Severity Distributions Across Models')
print(f'  H = {H:.4f}, p = {p_kw:.6f}')
print(f'  Interpretation: {"Significant (p<0.05)" if p_kw < 0.05 else "Not significant"}')

## 9. Early Failure Signal Analysis

In [ ]:
FAILURE_SIGNALS = {
    'loop_signal': ['again', 'retry', 'same', 'repeated', 'loop', 'replan'],
    'uncertainty_signal': ["i'm not sure", 'unclear', 'ambiguous', 'maybe', 'might'],
    'error_signal': ['error', 'failed', 'exception', 'cannot', 'unable'],
    'abandon_signal': ['give up', 'abandon', 'impossible', "can't", 'cannot complete'],
    'tool_failure_signal': ['bad request', '400', '429', '401', 'tool error'],
    'hallucination_signal': ['as i mentioned', 'as you know', 'i recall', 'i remember'],
}

def has_signal(record, keywords, n_steps=3):
    traj = record.get('trajectory', [])[:n_steps]
    text = ' '.join((s.get('action','') + ' ' + s.get('observation','')).lower() for s in traj)
    return any(kw in text for kw in keywords)

failures = [r for r in records if r.get('outcome') == 'failure']
successes = [r for r in records if r.get('outcome') != 'failure']

signal_data = []
for name, keywords in FAILURE_SIGNALS.items():
    fail_pct = sum(has_signal(r, keywords) for r in failures) / max(len(failures), 1) * 100
    succ_pct = sum(has_signal(r, keywords) for r in successes) / max(len(successes), 1) * 100
    signal_data.append({'Signal': name, 'In Failures': fail_pct, 'In Successes': succ_pct})

sig_df = pd.DataFrame(signal_data)
print(sig_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(signal_data))
width = 0.35
ax.bar([i - width/2 for i in x], sig_df['In Failures'], width, label='In Failures', color='#C44E52')
ax.bar([i + width/2 for i in x], sig_df['In Successes'], width, label='In Successes', color='#55A868')
ax.set_xticks(list(x))
ax.set_xticklabels(sig_df['Signal'], rotation=20, ha='right')
ax.set_ylabel('% of Records with Signal (first 3 steps)')
ax.set_title('Early Failure Signals: Presence in Failed vs Successful Trajectories', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../analysis/results/figures/early_signals.png', dpi=300, bbox_inches='tight')
plt.show()

## 10. Failure Rate by Task Type and Model

In [ ]:
TASK_TYPES = ['information_seeking', 'tool_use', 'planning', 'reasoning', 'multi_agent']
task_model_matrix = []
for model in MODELS:
    row = []
    for tt in TASK_TYPES:
        subset = df[(df['model'] == model) & (df['task_type'] == tt)]
        if len(subset) == 0:
            row.append(np.nan)
        else:
            row.append((subset['outcome'] == 'failure').mean() * 100)
    task_model_matrix.append(row)

tm_df = pd.DataFrame(task_model_matrix, index=MODELS,
                     columns=[tt.replace('_', ' ').title() for tt in TASK_TYPES])

fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(tm_df, annot=True, fmt='.1f', cmap='RdYlGn_r',
            linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Failure Rate (%)'},
            vmin=0, vmax=100)
ax.set_title('Failure Rate (%) by Task Type and Model', fontweight='bold', fontsize=14, pad=12)
ax.set_xlabel('Task Type')
ax.set_ylabel('Model')
plt.tight_layout()
plt.savefig('../analysis/results/figures/task_model_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## 11. Summary of Key Findings

In [ ]:
print('='*60)
print('AGENT FAILURE ATLAS — KEY FINDINGS')
print('='*60)

total = len(df)
overall_fail_rate = (df['outcome'] == 'failure').mean()
overall_recovery = df['recovered'].mean()
most_common_cat = df['failure_label'].value_counts().index[0]
most_common_subcat = df['failure_subcategory'].value_counts().index[0]

worst_model = stats_df.sort_values('Failure Rate', ascending=False)['Model'].iloc[0]
best_model = stats_df.sort_values('Failure Rate', ascending=True)['Model'].iloc[0]

print(f'\n1. Overall failure rate: {overall_fail_rate:.1%}')
print(f'2. Overall recovery rate: {overall_recovery:.1%}')
print(f'3. Most common failure category: {most_common_cat}')
print(f'4. Most common subcategory: {most_common_subcat}')
print(f'5. Highest failure rate model: {worst_model}')
print(f'6. Lowest failure rate model: {best_model}')
print(f'\n7. Safety failures (SAFE): 0% recovery rate — all are critical')
print(f'8. Execution loops (EXEC-IL): most severe category (avg severity 4.5)')
print(f'9. Planning failures most common in smaller models')
print(f'10. DeepSeek-R1-8B shows lowest reasoning failure rate (explicit CoT)')

print('='*60)